# 📊 Notebook 02 — ÚLOHY: Statistická analýza & EKC

---

## Jak na to
1. Spusť nejprve setup buňku
2. Pro každou úlohu napiš kód do buňky s `# TVŮJ KÓD ZDE`
3. Porovnej výsledek s očekávaným výstupem
4. Řešení je v `02_statistika_ekc_RESENI.ipynb`

## Předpoklady
- [ ] Dokončena Úloha 7 z Notebooku 01 (existuje `../output/ekc_analysis.csv`)

---
## ⚙️ Setup

Tuto buňku spusť vždy jako první — načte data a nastaví prostředí.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os, warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.3f}'.format)

# Načtení dat
df = pd.read_csv('../output/ekc_analysis.csv')
if 'log_gdp' not in df.columns:
    df['log_gdp'] = np.log(df['mean_gdp'])

income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
df['income_group'] = pd.Categorical(df['income_group'], categories=income_order, ordered=True)

print(f'Dataset načten: {len(df)} zemí')
print(f'Sloupce: {df.columns.tolist()}')
assert 180 <= len(df) <= 210, f"Neocekavany pocet zemi: {len(df)} (ocekavano ~199) — zkontroluj notebook 01!"

Dataset načten: 199 zemí
Sloupce: ['country', 'code', 'forest_1990', 'forest_2025', 'forest_change', 'mean_gdp', 'region', 'income_group', 'log_gdp', 'outlier_category']


---
## Úloha 1 — Grafická kontrola: Scatter plot HDP vs. změna lesa

> ℹ️ **Kód je připraven předem** — tvorba vizualizací v Pythonu není cílem tohoto cvičení. Spusť buňku níže a zkontroluj, zda graf odpovídá tvým očekáváním.

**Co by měl scatter plot ukazovat**: Přibližně 199 bodů (jedna země = jeden bod). Viditelný trend — body High income (zelené) jsou spíše nad nulou, body Low income (červené) spíše pod nulou. Svislá osa (forest_change) sahá přibližně od -35 do +17.

In [ ]:
# Grafická kontrola — kód je připraven předem
# Spusť tuto buňku a zkontroluj, že graf odpovídá tvým očekáváním

colors = {'Low income': '#d62728', 'Lower middle income': '#ff7f0e',
          'Upper middle income': '#1f77b4', 'High income': '#2ca02c'}

fig, ax = plt.subplots(figsize=(10, 6))
for group in ['Low income', 'Lower middle income', 'Upper middle income', 'High income']:
    mask = df['income_group'] == group
    ax.scatter(df[mask]['log_gdp'], df[mask]['forest_change'],
               label=group, color=colors[group], alpha=0.7, s=40)

ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, label='0 — hranice zalesňování')
ax.set_xlabel('log(HDP per capita)')
ax.set_ylabel('Změna lesního pokryvu 1990–2025 [pp]')
ax.set_title('EKC scatter plot: HDP vs. změna lesa (199 zemí)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# Kontrolní výpis
above_zero = (df['forest_change'] > 0).sum()
below_zero = (df['forest_change'] < 0).sum()
print(f'Bodů nad y=0 (zalesňování): {above_zero}')
print(f'Bodů pod y=0 (odlesňování): {below_zero}')

---
## Úloha 2: Pearsonova korelace

**Zadání**: Vypočítej Pearsonovu korelaci mezi `log_gdp` a `forest_change`:
1. Odstraň řádky kde chybí hodnoty (`.dropna()`)
2. Použij `stats.pearsonr(x, y)` → vrátí `(r, p_value)`
3. Vypiš:
   - Hodnotu r (zaokrouhlenou na 4 desetinná místa)
   - p-hodnotu
   - Zda je výsledek statisticky významný (p < 0.05)

> 💡 Import: `from scipy import stats`

**Očekávaný výstup (formát):**
```
Korelační koeficient r: 0.3625
p-hodnota:              0.0000
Statisticky významná:   ANO (p < 0.05)
```

In [3]:
# TVŮJ KÓD ZDE

# cor_data = df[['log_gdp', 'forest_change']].dropna()
# r, p_value = stats.pearsonr(..., ...)
# print(...)

> 📝 **Jak číst výsledek:**
> - **Směr** (`r` kladné/záporné): kladné = bohatší země → více lesa; záporné = opak
> - **Síla** (`|r|`): méně než 0.3 = slabá, 0.3–0.7 = střední, nad 0.7 = silná
> - **Průkaznost** (`p < 0.05`): vztah není náhodný, platí statisticky
> - **Zamysli se**: Proč může být korelace slabá, i když EKC platí? (Nápověda: EKC je **nelineární**!)

---
## Úloha 3: Polynomiální regrese (EKC křivka)

**Zadání**:
1. Připrav pole `x = log_gdp` a `y = forest_change` (bez NaN)
2. Fit kvadratickou regresi: `np.polyfit(x, y, 2)` → `coeffs_quad` (koeficienty [a, b, c])
3. Výpočet **bodu zlomu**: `x_vertex = -b / (2 * a)` → zpětný převod: `gdp_inflection = np.exp(x_vertex)`
4. Výpočet **R²** pro kvadratický model (vzorec: `1 - SS_res/SS_tot`)
5. Vypiš bod zlomu a R²
6. Spusť kontrolní vizualizaci z buňky níže — EKC křivka je připravena předem

> 💡 Koeficienty z `np.polyfit(x, y, 2)` jsou `[a, b, c]` kde `y = a·x² + b·x + c`
>
> 💡 `np.polyval(coeffs, x)` spočítá predikované hodnoty
>
> 💡 **Proč `-b/(2a)`?** Kvadratická funkce `y = ax² + bx + c` má bod zlomu
> tam kde derivace `2ax + b = 0`, tedy `x = -b/(2a)`. Přes `np.exp()` převedeš
> `log(HDP)` zpět na USD — a dostaneš bod zlomu jako HDP na obyvatele.

> ✅ **Dva sanity checky po výpočtu**:
> - Koeficient `a` by měl být **záporný** → křivka má tvar ∩ (to je EKC). Pokud `a > 0`, křivka je ∪ a výsledek nemá smysl — zkontroluj, že `x = log_gdp` (ne samotné GDP bez logaritmu).
> - Bod zlomu by měl vyjít přibližně **$5 000–$500 000/os.** Výsledky mimo tento rozsah naznačují chybu ve výpočtu bodu zlomu.

**Očekávaný výstup (formát):**
```
Koeficienty: a=-0.36252, b=7.91240, c=-41.49626
Bod zlomu:   $54,885/os.
R²:          0.1456
```

In [4]:
# TVŮJ KÓD ZDE — kroky 1-5

# reg_data = df[['log_gdp', 'forest_change']].dropna()
# x = ...
# y = ...

# coeffs_quad = np.polyfit(x, y, 2)
# a, b, c = coeffs_quad

# x_vertex = -b / (2 * a)
# gdp_inflection = np.exp(x_vertex)

# y_pred = np.polyval(coeffs_quad, x)
# ss_res = ...
# ss_tot = ...
# r_squared = 1 - ss_res / ss_tot

# print(f'Bod zlomu: ${gdp_inflection:,.0f}/os.')
# print(f'R² = {r_squared:.4f}')

In [ ]:
# Grafická kontrola — EKC křivka (kód připraven předem)
# Spusť tuto buňku až po dokončení kroků 1–5 výše
# (musí existovat: coeffs_quad, x_vertex, gdp_inflection)

reg_data = df[['log_gdp', 'forest_change']].dropna()
x_plot = reg_data['log_gdp'].values
y_plot = reg_data['forest_change'].values

fig, ax = plt.subplots(figsize=(10, 6))

for group in ['Low income', 'Lower middle income', 'Upper middle income', 'High income']:
    mask = df['income_group'] == group
    ax.scatter(df[mask]['log_gdp'], df[mask]['forest_change'],
               label=group, color=colors[group], alpha=0.6, s=30)

x_line = np.linspace(x_plot.min(), x_plot.max(), 200)
y_line = np.polyval(coeffs_quad, x_line)
ax.plot(x_line, y_line, 'k-', linewidth=2, label='EKC křivka (deg=2)')

ax.axvline(x=x_vertex, color='purple', linestyle=':', linewidth=1.5,
           label=f'Bod zlomu: ${gdp_inflection:,.0f}/os.')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

ax.set_xlabel('log(HDP per capita)')
ax.set_ylabel('Změna lesního pokryvu 1990–2025 [pp]')
ax.set_title('EKC model: Kvadratická regrese s bodem zlomu')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

> 📝 **Jak číst výsledek:**
> - **Znaménko `a`** (první koeficient): záporné = křivka ve tvaru ∩ (EKC teorie potvrzena)
> - **Bod zlomu v USD**: při jakém HDP se trend obrací? Je to hodnota dostupná pro chudé země?
> - **R²**: kolik procent variance model vysvětluje? Kvadratický model lepší než lineární?
> - **Závěr**: Leží většina zemí pod nebo nad bodem zlomu?

> 📊 **Power BI — vizualizace 1 (Q1 — EKC scatter)**: Bodový graf (Scatter) → osa X: `log_gdp`, osa Y: `forest_change`, barvy: `income_group`. Zdroj: `ekc_analysis.csv`. Viz **PowerBI_pruvodce.md, část 4**.
>
> 📊 **Power BI — vizualizace 2 (Q1 — příjmové skupiny)**: Skupinový sloupcový graf → osa X: `income_group`, osa Y: průměr `forest_change`. Zdroj: `ekc_analysis.csv`. Viz **PowerBI_pruvodce.md, část 4**.

---
## Úloha 4: Test hypotézy Q1

**Zadání**: Otestuj hypotézu pomocí **Mann-Whitney U testu**:
- H0: Změna lesa u High income a Low income zemí pochází ze stejné distribuce
- H1: High income země mají **vyšší** změnu lesa (jednostranný test)

Kroky:
1. Vyfiltruj `forest_change` pro `High income` a pro `Low income`
2. Vypiš průměr a počet zemí v obou skupinách — pomůže to při interpretaci výsledku
3. Spusť `stats.mannwhitneyu(high, low, alternative='greater')`
4. Vypiš U statistiku, p-hodnotu a závěr
5. **Bonus**: Spusť t-test: Je průměrná změna lesa High income zemí > 0?
   → `stats.ttest_1samp(high_income, popmean=0, alternative='greater')`

> 💡 **Proč `alternative='greater'`?** Parametr říká, že testujeme **jednostrannou** hypotézu: zajímá nás jen jestli High income je *větší* než Low income — ne jen "jiné". Přesně odpovídá naší H1. Dvoustranný test (`alternative='two-sided'`) by byl konzervativnější a hůře by odhalil tento konkrétní směrový vztah.

**Očekávaný výstup (formát):**
```
Mann-Whitney U = 1594, p = 0.0000
Závěr: H1 POTVRZENA

t-test (High income > 0): t=2.7944, p=0.0033
Závěr: High income PRŮMĚRNĚ ZALESŇUJÍ
```

In [6]:
# TVŮJ KÓD ZDE

# high_income_change = df[df['income_group'] == 'High income']['forest_change'].dropna()
# low_income_change  = ...

# u_stat, p_mw = stats.mannwhitneyu(...)
# print(...)

> 📝 **Jak číst výsledek:**
> - **Jednostranný test** (`alternative='greater'`): testujeme konkrétní směr — H1: High > Low
> - **p < 0.05** → H1 potvrzena: High income země skutečně zalesňují více
> - **p >= 0.05** → H0 nezamítnuta: nemáme dostatečný důkaz pro H1
> - **Závěr pro Q1**: Potvrzuje výsledek EKC hypotézu o vlivu bohatství na lesy?

---
## Úloha 5: Regionální analýza (Q2)

**Zadání**:
1. Spočítej `groupby('region')` statistiky pro `forest_change`: n, mean, median, std
2. Seřaď podle `mean_change` vzestupně
3. Spusť **Kruskal-Wallis test** přes všechny regiony:
   - Vytvoř list polí: `[df[df['region']==r]['forest_change'].dropna().values for r in regions]`
   - `stats.kruskal(*groups)` → `(H, p)`
4. Vypiš závěr (jsou regionální rozdíly statisticky významné?)

> 💡 `*list` rozbalí list jako argumenty funkce: `stats.kruskal(*[array1, array2, array3])`
>    je totéž jako `stats.kruskal(array1, array2, array3)`

> 💡 **Nápověda k pojmenování sloupců**: Použij pojmenovaný formát agregace místo obyčejného seznamu — jinak sloupce nebudou mít správná jména pro Power BI export v Úloze 8:
> ```python
> .agg(n='count', mean_change='mean', median_change='median', std_change='std')
> ```

**Očekávaný výstup (formát):**
```
                             n  mean_change  ...
region
Sub-Saharan Africa          46       -5.775  ...
Latin America & Caribbean   35       -3.204  ...
...
Europe & Central Asia       55        2.747  ...

Kruskal-Wallis: H=62.5337, p=0.000000
Regionální rozdíly jsou STATISTICKY VÝZNAMNÉ
```

In [7]:
# TVŮJ KÓD ZDE

# regional_summary = ...
# print(regional_summary)

# region_groups = [...]
# region_groups = [g for g in region_groups if len(g) > 0]
# h_stat, p_kw = stats.kruskal(*region_groups)
# print(...)

> 📝 **Jak číst výsledek:**
> - Kruskal-Wallis testuje, zda **aspoň jeden** region se liší od ostatních
> - Statisticky významný výsledek = rozdíly jsou příliš velké na to, aby byly náhodné
> - Test **neřekne** které regiony se liší — to by vyžadovalo post-hoc analýzu
> - **Závěr pro Q2**: Závisí změna lesa na geografické poloze nad rámec ekonomiky?

> 📊 **Power BI — vizualizace 3 (Q2 — regiony)**: Sloupcový graf (vodorovný) → osa Y: `region`, osa X: `mean_change`, barvy: `region`. Zdroj: `regional_summary.csv`. Viz **PowerBI_pruvodce.md, část 4**.

---
## Úloha 6: Outliers — Paradoxní země (Q3)

**Zadání**:
1. Vypočítej Q75 a Q25 pro `mean_gdp` (`.quantile(0.75)` a `.quantile(0.25)`)
2. Filtruj **bohaté odlesňovatelé**: `mean_gdp > Q75` AND `forest_change < 0`
3. Filtruj **chudé zalesňovatelé**: `mean_gdp < Q25` AND `forest_change > 0.5`
4. Vypiš obě skupiny zemí s jejich `income_group`, `region`, `mean_gdp`, `forest_change`
5. Přidej do `df` sloupec `outlier_category` s hodnotami:
   `'rich_deforester'`, `'poor_reforester'`, `'expected_positive'`, `'expected_negative'`

> ℹ️ Krok 5 je nutný pro Úlohu 7 (Forest Policy) i Úlohu 8 (export).
> Použij `df.apply(classify, axis=1)` kde `classify` je pomocná funkce vracející kategorii pro každý řádek.

> 💡 **Logika klasifikace — jak přiřadit všechna 4 pásma:**
> - `'rich_deforester'` → mean_gdp > Q75 **AND** forest_change < 0
> - `'poor_reforester'` → mean_gdp < Q25 **AND** forest_change > 0.5
> - `'expected_positive'` → vše ostatní s forest_change ≥ 0 (zalesňování — odpovídá EKC)
> - `'expected_negative'` → vše ostatní s forest_change < 0 (odlesňování — odpovídá EKC)

**Očekávaný výstup (formát):**
```
Bohaté odlesňovatelé (8):
   Brunei, South Korea, Cayman Islands, Sweden, Norway, Belgium, Australia, Japan

Chudé zalesňovatelé (6):
   Rwanda, Ghana, Nepal, Uzbekistan, India, Kyrgyzstan
```

In [8]:
# TVŮJ KÓD ZDE

# q75_gdp = df['mean_gdp'].quantile(0.75)
# q25_gdp = ...

# rich_deforesters = df[...]
# poor_reforesters = df[...]

# print('Bohaté odlesňovatelé:')
# print(rich_deforesters[...].to_string(index=False))

> 📊 **Power BI — vizualizace 4 (Q3 — mapa světa)**: Choropleth mapa → umístění: `code`, barva: `forest_change`. Zdroj: `ekc_analysis.csv`. Viz **PowerBI_pruvodce.md, část 4**.
>
> 📊 **Power BI — vizualizace 5 (Q3 — paradoxní země)**: Tabulka → sloupce: `country`, `income_group`, `forest_change`, `outlier_category`, `policy_score`. Zdroj: `outliers_with_policy.csv`. Viz **PowerBI_pruvodce.md, část 4**.

---
## Úloha 7: Forest Policy — Vysvětlují data o politice paradoxní země?

**Zadání**: Načti a analyzuj Forest Policy data pro paradoxní země z Úlohy 6:

1. Načti `../../data_raw/Forest_Policy_Legislation.csv` do `fp_raw`
   - Použij `on_bad_lines='skip'`
2. Vyber sloupce: `iso3`, `National policies supporting SFM`, `National legislations supporting SFM`, `National platform for stakeholder participation`
3. Přejmenuj je: `code`, `has_policy`, `has_legislation`, `has_platform`
4. Převeď `yes`/`no` na `True`/`False` pomocí `.str.lower().map({'yes': True, 'no': False})`
5. Přidej sloupec `policy_score = součet True hodnot` v řádku (`.sum(axis=1)`)
6. Propoj s `df[df['outlier_category'].isin(['rich_deforester', 'poor_reforester'])]` přes `code`
7. Porovnej průměrný `policy_score` pro `rich_deforester` vs. `poor_reforester`
8. **Závěr**: Mají chudé zalesňovatelé silnější lesní politiku?

> 💡 **Proč `on_bad_lines='skip'`?** CSV soubor obsahuje volné textové popisy v některých buňkách. Tyto texty mívají čárky uvnitř, takže pandas napočítá víc sloupců než je v záhlaví. Místo chyby `ParserError` takové řádky přeskočíme. Ověř počet načtených řádků: `print(f'Načteno: {len(fp_raw)} řádků')` — mělo by jich být víc než 100.

> 💡 Nápověda ke kroku 4:
> ```python
> for col in ['has_policy', 'has_legislation', 'has_platform']:
>     fp[col] = fp[col].str.strip().str.lower().map({'yes': True, 'no': False})
> ```
>
> 💡 Nápověda ke kroku 5:
> ```python
> fp['policy_score'] = fp[['has_policy', 'has_legislation', 'has_platform']].sum(axis=1)
> ```

**Očekávaný výstup (formát):**
```
Forest Policy data: 236 zemí
Průměrný policy score: 2.30 (max = 3)

                  n   mean  median
outlier_category
poor_reforester   6  2.833   3.000
rich_deforester   8  2.750   3.000
```

> 🔍 **Klíčový závěr**: Obě skupiny mají vysoký policy score (~2.75–2.83). Lesní politika sama o sobě neodlišuje paradoxní země — jiné faktory (geografie, hustota obyvatelstva, historický vývoj) jsou pravděpodobně důležitější.

In [9]:
# TVŮJ KÓD ZDE

# Krok 1-2: Načtení a výběr sloupců
# fp_raw = pd.read_csv('../../data_raw/Forest_Policy_Legislation.csv', on_bad_lines='skip')
# fp = fp_raw[['iso3', ...]].copy()
# fp.columns = ['code', 'has_policy', 'has_legislation', 'has_platform']

# Krok 3-4: Konverze a policy score
# for col in [...]:
#     fp[col] = fp[col].str.strip().str.lower().map({'yes': True, 'no': False})
# fp['policy_score'] = ...

# Krok 5-6: Merge s outliers
# outlier_policy = pd.merge(...)

# Krok 7: Srovnání
# print(outlier_policy.groupby('outlier_category')['policy_score'].mean())

# Krok 8: Závěr
# print('Závěr: ...')


---
## Úloha 8: Export výstupů pro Power BI

**Zadání**: Exportuj 5 souborů do složky `../output/`:
1. `ekc_analysis.csv` — hlavní dataset `df` (s outlier_category sloupcem)
2. `ekc_regression_curve.csv` — 100 bodů pro EKC křivku:
   - `x_line = np.linspace(df['log_gdp'].min(), df['log_gdp'].max(), 100)`
   - `y_pred = np.polyval(coeffs_quad, x_line)`
   - DataFrame se sloupci: `log_gdp_fit`, `gdp_fit = np.exp(x_line)`, `forest_pred`
3. `regional_summary.csv` — regionální statistiky (z Úlohy 5)
4. `outliers.csv` — paradoxní země (z Úlohy 6)
5. `outliers_with_policy.csv` — paradoxní země obohacené o policy data (z Úlohy 7)
   - Použi `outlier_policy` z Úlohy 7 a vyber sloupce:
     `country`, `income_group`, `region`, `mean_gdp`, `forest_change`,
     `outlier_category`, `has_policy`, `has_legislation`, `policy_score`

> ℹ️ Soubory 3–5 vyžadují proměnné z předchozích úloh: `coeffs_quad` (Ú3),
> `regional_summary` (Ú5), `outlier_policy` (Ú7) — spusť je nejdřív.

**Očekávaný výstup:**
```
✅ ekc_analysis.csv          (199 řádků)
✅ ekc_regression_curve.csv  (100 bodů pro EKC křivku)
✅ regional_summary.csv      (7 regionů)
✅ outliers.csv              (14 zemí — 8 rich_deforester + 6 poor_reforester)
✅ outliers_with_policy.csv
```

> ✅ **Ověření**: Otevři složku `../output/` v Průzkumníku souborů a zkontroluj, že všechny soubory existují a mají nenulovou velikost.

In [10]:
# TVŮJ KÓD ZDE

# os.makedirs('../output', exist_ok=True)

# df.to_csv(...)

# x_line = np.linspace(...)
# y_quad_line = np.polyval(coeffs_quad, x_line)
# ekc_curve = pd.DataFrame({'log_gdp_fit': x_line, 'gdp_fit': np.exp(x_line), 'forest_pred': y_quad_line})
# ekc_curve.to_csv(...)

# ...

# print('Všechny soubory exportovány!')

---
## ✅ Hotovo?

Zkontroluj:
- [ ] Scatter plot HDP vs. změna lesa nakreslen (Úloha 1)
- [ ] Korelace r a p-hodnota vypočítány (Úloha 2)
- [ ] EKC křivka zobrazena se bodem zlomu (Úloha 3)
- [ ] Mann-Whitney test proveden a interpretován (Úloha 4)
- [ ] Kruskal-Wallis test proveden (Úloha 5)
- [ ] Paradoxní země identifikovány, sloupec `outlier_category` přidán (Úloha 6)
- [ ] Forest Policy analýza provedena (Úloha 7)
- [ ] 5 souborů exportováno do `../output/` (Úloha 8)

**Řešení**: `02_statistika_ekc_RESENI.ipynb`
**Pokračuj**: [`../PowerBI_pruvodce.md`](../PowerBI_pruvodce.md)